In [19]:
import numpy as np
import pandas as pd
import statsmodels.api as sm


def select_ar_lag(series, max_p, criterion):
    df = pd.DataFrame({"y": series})

    # Create all lags up to max_p
    for i in range(1, max_p + 1):
        df[f"lag_{i}"] = df["y"].shift(i)
    
    df = df.dropna()  # ONE common sample
    
    y = df["y"]
    
    best_p = None
    best_ic = np.inf
    
    for p in range(1, max_p + 1):
        X = df[[f"lag_{i}" for i in range(1, p + 1)]]
        X = sm.add_constant(X)
        
        model = sm.OLS(y, X).fit()
        
        ic = model.bic if criterion == "bic" else model.aic
        
        if ic < best_ic:
            best_ic = ic
            best_p = p
    
    return best_p

In [20]:
df = pd.read_csv("../../data/fred_md.csv")
df["sasdate"] = pd.to_datetime(df["sasdate"])
df = df[(df["sasdate"].dt.year < 2020) & (df["sasdate"].dt.year >= 1959)]
bic_p = {}
aic_p = {}

for col in df.columns[1:]:  # Skip date column
    best_p = select_ar_lag(df[col], max_p=12, criterion="bic")
    bic_p[col] = best_p

In [21]:
df_qd = pd.read_csv("../../data/fred_qd_X.csv")
df_qd["sasdate"] = pd.to_datetime(df_qd["sasdate"])
df_qd = df_qd[(df_qd["sasdate"].dt.year < 2020) & (df_qd["sasdate"].dt.year >= 1959)]

for col in df_qd.columns[1:]:  # Skip date column
    best_p = select_ar_lag(df_qd[col], max_p=5, criterion="bic")
    bic_p[col] = best_p

In [17]:
bic_p

{'RPI_t': 1,
 'W875RX1_t': 1,
 'DPCERA3M086SBEA_t': 1,
 'CMRMTSPLx_t': 1,
 'RETAILx_t': 2,
 'INDPRO_t': 1,
 'IPFPNSS_t': 4,
 'IPFINAL_t': 1,
 'IPCONGD_t': 1,
 'IPDCONGD_t': 1,
 'IPNCONGD_t': 3,
 'IPBUSEQ_t': 1,
 'IPMAT_t': 1,
 'IPDMAT_t': 1,
 'IPNMAT_t': 2,
 'IPMANSICS_t': 1,
 'IPB51222S_t': 2,
 'IPFUELS_t': 1,
 'CUMFNS_t': 2,
 'HWI_t': 6,
 'HWIURATIO_t': 4,
 'CLF16OV_t': 1,
 'CE16OV_t': 1,
 'UNRATE_t': 1,
 'UEMPMEAN_t': 3,
 'UEMPLT5_t': 1,
 'UEMP5TO14_t': 1,
 'UEMP15OV_t': 4,
 'UEMP15T26_t': 1,
 'UEMP27OV_t': 1,
 'CLAIMSx_t': 1,
 'PAYEMS_t': 2,
 'USGOOD_t': 2,
 'CES1021000001_t': 1,
 'USCONS_t': 1,
 'MANEMP_t': 2,
 'DMANEMP_t': 1,
 'NDMANEMP_t': 2,
 'SRVPRD_t': 1,
 'USTPU_t': 1,
 'USWTRADE_t': 2,
 'USTRADE_t': 2,
 'USFIRE_t': 2,
 'USGOVT_t': 3,
 'CES0600000007_t': 1,
 'AWOTMAN_t': 1,
 'AWHMAN_t': 2,
 'HOUST_t': 1,
 'HOUSTNE_t': 1,
 'HOUSTMW_t': 1,
 'HOUSTS_t': 1,
 'HOUSTW_t': 1,
 'AMDMNOx_t': 1,
 'AMDMUOx_t': 1,
 'BUSINVx_t': 1,
 'ISRATIOx_t': 1,
 'M1SL_t': 8,
 'M2SL_t': 8,
 'M2REAL_t

In [22]:
df_p = pd.DataFrame(list(bic_p.items()), columns=["variable", "lag"])
df_p.to_csv("bic_lags.csv", index=False)